## Working with Complex Data

Databricks : https://dbc-0b69ef93-5247.cloud.databricks.com/editor/notebooks/2460720313805616?o=7474657872577658

 JSON Strings vs Structs

Semi-structured data often arrives as JSON strings. Convert it early:

    • JSON strings require parsing overhead and waste memory
    • Structs give type safety and better performance
    • Convert early using from_json() with a schema

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import DoubleType, LongType, MapType, StructField, StructType

# 1. Inicializando a SparkSession (caso já não exista no seu notebook)
spark = SparkSession.builder.appName("JsonVsStructsExample").getOrCreate()

# Dados de exemplo fornecidos
data = [
    (1, '{"user_id": 100, "features": {"score": 0.8}}'),
    (2, '{"user_id": 101, "features": {"score": 0.95}}'),
]

df = spark.createDataFrame(data, ["id", "json_data"])

print("--- DataFrame Original (String JSON) ---")
df.printSchema()
df.show(truncate=False)

--- DataFrame Original (String JSON) ---
root
 |-- id: long (nullable = true)
 |-- json_data: string (nullable = true)

+---+---------------------------------------------+
|id |json_data                                    |
+---+---------------------------------------------+
|1  |{"user_id": 100, "features": {"score": 0.8}} |
|2  |{"user_id": 101, "features": {"score": 0.95}}|
+---+---------------------------------------------+



In [0]:
from pyspark.sql.types import StringType

# 2. Definindo o schema explícito que corresponde à estrutura do JSON
schema = StructType(
    [
        StructField("user_id", LongType(), True),
        # 'features' é um dicionário/mapa contendo chaves String e valores Double
        StructField("features", MapType(StringType(), DoubleType()), True),
    ]
)

# 3. Conversão antecipada de string JSON para Struct usando from_json()
df_parsed = df.withColumn("parsed", from_json(col("json_data"), schema))

print("\n--- DataFrame Após a Conversão (StructType) ---")
df_parsed.printSchema()
df_parsed.show(truncate=False)


--- DataFrame Após a Conversão (StructType) ---
root
 |-- id: long (nullable = true)
 |-- json_data: string (nullable = true)
 |-- parsed: struct (nullable = true)
 |    |-- user_id: long (nullable = true)
 |    |-- features: map (nullable = true)
 |    |    |-- key: string
 |    |    |-- value: double (valueContainsNull = true)

+---+---------------------------------------------+----------------------+
|id |json_data                                    |parsed                |
+---+---------------------------------------------+----------------------+
|1  |{"user_id": 100, "features": {"score": 0.8}} |{100, {score -> 0.8}} |
|2  |{"user_id": 101, "features": {"score": 0.95}}|{101, {score -> 0.95}}|
+---+---------------------------------------------+----------------------+



In [0]:

# 4. Acessando os campos estruturados de forma segura e performática
# Em vez de reanalisar o JSON a cada acesso, podemos navegar pelas chaves do Struct diretamente
df_final = df_parsed.select(
    col("id"),
    col("parsed.user_id").alias("user_id"),
    col("parsed.features")["score"].alias(
        "score"
    ),  # Acessando o valor dentro do mapa
)

print("\n--- Extraindo Campos Finais com Type Safety ---")
df_final.show()


--- Extraindo Campos Finais com Type Safety ---
+---+-------+-----+
| id|user_id|score|
+---+-------+-----+
|  1|    100|  0.8|
|  2|    101| 0.95|
+---+-------+-----+



C. Working with Struct Types
Struct types maintain a logical grouping of related fields - like a table within a column:
    • Struct fields can be accessed and operated on using dot notation or the getField() column function
Given input data with struct column 'user':

user: struct&lt;name:string, age:int, scores:array&lt;double&gt;&gt;;

```
# Accessing struct fields
df.select(
    col("user.name"), # Direct field access
    col("user").getField("age"), # Alternative access
    col("user.scores")[0].alias("first_score") # Nested access
)
```

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import ArrayType, DoubleType, IntegerType, StringType, StructField, StructType

# 1. Inicializando a SparkSession
spark = SparkSession.builder.appName("WorkingWithStructTypesExample").getOrCreate()

# Definindo o schema explícito com StructType contendo strings, inteiros e um array
schema = StructType([
    StructField("user", StructType([
        StructField("name", StringType(), True),
        StructField("age", IntegerType(), True),
        StructField("scores", ArrayType(DoubleType()), True)
    ]), True)
])

# Dados de exemplo simulando a estrutura solicitada
data = [
    ({"name": "Alice", "age": 30, "scores": [85.5, 90.0, 92.5]},),
    ({"name": "Bob", "age": 25, "scores": [78.0, 82.5, 88.0]},)
]

df = spark.createDataFrame(data, schema)

print("--- Schema Original com StructType ---")
df.printSchema()
df.show(truncate=False)

--- Schema Original com StructType ---
root
 |-- user: struct (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- age: integer (nullable = true)
 |    |-- scores: array (nullable = true)
 |    |    |-- element: double (containsNull = true)

+-------------------------------+
|user                           |
+-------------------------------+
|{Alice, 30, [85.5, 90.0, 92.5]}|
|{Bob, 25, [78.0, 82.5, 88.0]}  |
+-------------------------------+



In [0]:
# 2. Acessando os campos do Struct de diferentes maneiras
df_extracted = df.select(
    # Acesso direto usando a notação de ponto (mais limpo e idiomático)
    col("user.name").alias("user_name"),
    
    # Acesso alternativo usando getField() (equivalente à notação de ponto)
    col("user").getField("age").alias("user_age"),
    
    # Acesso a estruturas aninhadas (pegando o primeiro elemento do array 'scores')
    col("user.scores")[0].alias("first_score")
)

print("\n--- DataFrame Após Extração de Campos ---")
df_extracted.show()


--- DataFrame Após Extração de Campos ---
+---------+--------+-----------+
|user_name|user_age|first_score|
+---------+--------+-----------+
|    Alice|      30|       85.5|
|      Bob|      25|       78.0|
+---------+--------+-----------+




### D. The explode() Function

Unnesting arrays while preserving other columns - one row per array element:
Input: DataFrame with array column:

```
+---+----------------+
|id |items           |
+---+----------------+
|1  |["a", "b", "c"] |
|2  |["x", "y"]      |
+---+----------------+

```

df.select("id", explode("items").alias("item"))
```
Results in...
+---+----+
|id |item|
+---+----+
|1  |a   |
|1  |b   |
|1  |c   |
|2  |x   |
|2  |y   |
+---+----+
```

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode

# 1. Inicializando a SparkSession
spark = SparkSession.builder.appName("ExplodeFunctionExample").getOrCreate()

# 2. Dados de exemplo correspondentes ao cenário fornecido
data = [
    (1, ["a", "b", "c"]),
    (2, ["x", "y"])
]

df = spark.createDataFrame(data, ["id", "items"])

print("--- DataFrame Original com Coluna Array ---")
df.printSchema()

df.show(truncate=False)

--- DataFrame Original com Coluna Array ---
root
 |-- id: long (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: string (containsNull = true)

+---+---------+
|id |items    |
+---+---------+
|1  |[a, b, c]|
|2  |[x, y]   |
+---+---------+



In [0]:
# 3. Aplicando a função explode() para expandir os arrays em linhas separadas
df_exploded = df.select(
    col("id"),
    explode(col("items")).alias("item")
)

print("\n--- Resultado após o uso de explode() ---")
df_exploded.show()


--- Resultado após o uso de explode() ---
+---+----+
| id|item|
+---+----+
|  1|   a|
|  1|   b|
|  1|   c|
|  2|   x|
|  2|   y|
+---+----+



## E. Common Array Column Operations

Common functions for array column manipulation:

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import array_contains, array_distinct, col, element_at, size

# 1. Inicializando a SparkSession
spark = SparkSession.builder.appName("ArrayOperationsExample").getOrCreate()

# 2. Dados de exemplo com arrays contendo elementos duplicados e diferentes tamanhos
data = [
    (1, ["apple", "banana", "apple", "orange"]),
    (2, ["grape", "pear"]),
    (3, [])
]

df = spark.createDataFrame(data, ["id", "items"])

print("--- DataFrame Original ---")
df.show(truncate=False)

--- DataFrame Original ---
+---+------------------------------+
|id |items                         |
+---+------------------------------+
|1  |[apple, banana, apple, orange]|
|2  |[grape, pear]                 |
|3  |[]                            |
+---+------------------------------+



In [0]:
# 3. Aplicando operações comuns em arrays
from pyspark.sql.functions import lit, try_element_at

df_operations = df.select(
    col("id"),
    col("items"),
    # size(): Retorna o número de elementos no array
    size(col("items")).alias("array_size"),
    # array_contains(): Retorna booleano indicando se o elemento existe no array
    array_contains(col("items"), "apple").alias("has_apple"),
    # try_element_at(): Retorna o elemento em uma posição específica (base 1 no Spark!)
    # Diferente do element_at(), retorna NULL em vez de erro quando o índice é inválido (ex.: array vazio)
    try_element_at(col("items"), lit(1)).alias("first_element"),
    # array_distinct(): Remove elementos duplicados do array
    array_distinct(col("items")).alias("distinct_items")
)

print("\n--- Resultado das Operações com Arrays ---")
df_operations.show(truncate=False)


--- Resultado das Operações com Arrays ---
+---+------------------------------+----------+---------+-------------+-----------------------+
|id |items                         |array_size|has_apple|first_element|distinct_items         |
+---+------------------------------+----------+---------+-------------+-----------------------+
|1  |[apple, banana, apple, orange]|4         |true     |apple        |[apple, banana, orange]|
|2  |[grape, pear]                 |2         |false    |grape        |[grape, pear]          |
|3  |[]                            |0         |false    |NULL         |[]                     |
+---+------------------------------+----------+---------+-------------+-----------------------+



### F. Aggregating to Collections

Collection aggregates are the inverse of explode() - they gather rows into arrays:

    • collect_list() is an aggregate function that gathers all values from a column into an array - commonly used with groupBy() to build arrays of related values within each group
    • collect_set() works similarly but removes duplicates, producing arrays of unique values

# Get a list (array) of products for each region

```
df.groupBy("region") \
  .agg(
    collect_list("product").alias("all_products"),    # keeps duplicates
    collect_set("product").alias("unique_products")   # unique only
  )
``

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import collect_list, collect_set, col

# 1. Inicializando a SparkSession
spark = SparkSession.builder.appName("AggregatingToCollectionsExample").getOrCreate()

# 2. Dados de exemplo: transações de produtos por região (com duplicadas)
data = [
    ("North", "Laptop"),
    ("North", "Mouse"),
    ("North", "Laptop"), # Produto duplicado na mesma região
    ("South", "Monitor"),
    ("South", "Keyboard")
]

df = spark.createDataFrame(data, ["region", "product"])

print("--- DataFrame Original (Linhas por Transação) ---")
df.show(truncate=False)

--- DataFrame Original (Linhas por Transação) ---
+------+--------+
|region|product |
+------+--------+
|North |Laptop  |
|North |Mouse   |
|North |Laptop  |
|South |Monitor |
|South |Keyboard|
+------+--------+



In [0]:


# 3. Agrupando por região e coletando os produtos em arrays
df_aggregated = df.groupBy("region").agg(
    # collect_list() mantém todos os valores, incluindo duplicadas
    collect_list("product").alias("all_products"),
    # collect_set() remove duplicadas, gerando apenas valores únicos
    collect_set("product").alias("unique_products")
)

print("\n--- Resultado após Agregação com collect_list() e collect_set() ---")
df_aggregated.show(truncate=False)


--- Resultado após Agregação com collect_list() e collect_set() ---
+------+-----------------------+-------------------+
|region|all_products           |unique_products    |
+------+-----------------------+-------------------+
|North |[Laptop, Laptop, Mouse]|[Laptop, Mouse]    |
|South |[Monitor, Keyboard]    |[Monitor, Keyboard]|
+------+-----------------------+-------------------+



### G. Best Practices for Complex Data Types

Considerations when working with nested structures:

    • explode() on large arrays can cause data explosion - 1 row becomes N rows, which can overwhelm memory and processing
    • collect_list() / collect_set() can be memory-intensive when groups are large, as they hold all values per group in memory
    • Consider collect_set() when duplicates aren't needed and order doesn't matter:

        ◦ Uses less memory because it removes duplicates
        ◦ Less shuffle overhead as duplicate values don't need to be collected